# 01 - Exploratory Data Analysis (EDA)
## COD-CRM: Order Risk Prediction for Algerian E-Commerce

This notebook explores the Olist Brazilian E-Commerce dataset, adapted to simulate
an Algerian COD (Cash-on-Delivery) CRM context.

### Dataset
- Source: [Kaggle - Brazilian E-Commerce by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
- ~100K orders with delivery status, customer, product, and payment data

### Goals
1. Understand the data structure and quality
2. Analyze delivery success/failure patterns
3. Identify key features for the risk prediction model
4. Explore regional, temporal, and product patterns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path
sys.path.insert(0, str(Path('.').resolve().parent))
from data.mapping import STATE_TO_WILAYA, STATUS_MAPPING, CATEGORY_TRANSLATION, BRL_TO_DZD

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully!')

## 1. Load the Data
Make sure you have downloaded the Olist dataset and placed the CSV files in `data/olist/`

In [ ]:
DATA_DIR = Path('../data/olist')

# Load all datasets
orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')

print(f'Orders:    {orders.shape}')
print(f'Customers: {customers.shape}')
print(f'Items:     {items.shape}')
print(f'Products:  {products.shape}')
print(f'Payments:  {payments.shape}')
print(f'Reviews:   {reviews.shape}')

## 2. Order Status Distribution
Understanding the target variable: delivery success vs failure

In [ ]:
# Map statuses to CRM format
orders['crm_status'] = orders['order_status'].map(STATUS_MAPPING)
orders['is_delivered'] = (orders['crm_status'] == 'delivered').astype(int)

print('=== Order Status Distribution ===')
print(orders['order_status'].value_counts())
print(f'\nDelivery Success Rate: {orders["is_delivered"].mean():.1%}')
print(f'Failure Rate: {1 - orders["is_delivered"].mean():.1%}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

orders['order_status'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Order Status Distribution (Raw)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

orders['crm_status'].value_counts().plot(kind='bar', ax=axes[1], color=['green', 'red', 'orange', 'gray', 'blue'])
axes[1].set_title('CRM Status Distribution (Mapped)')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Merge All Tables

In [ ]:
# Merge orders with customers
df = orders.merge(customers, on='customer_id', how='left')

# Aggregate items per order
items_agg = items.groupby('order_id').agg(
    n_items=('order_item_id', 'count'),
    subtotal=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
).reset_index()

# Get product category per order
items_products = items.merge(products[['product_id', 'product_category_name', 'product_weight_g']], on='product_id', how='left')
primary_cat = items_products.groupby('order_id').agg(
    product_category=('product_category_name', 'first'),
    avg_weight_kg=('product_weight_g', lambda x: x.mean() / 1000),
).reset_index()

# Merge all
df = df.merge(items_agg, on='order_id', how='left')
df = df.merge(primary_cat, on='order_id', how='left')

# Payments
pay_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', 'first'),
).reset_index()
df = df.merge(pay_agg, on='order_id', how='left')

# Reviews
rev_agg = reviews.groupby('order_id').agg(review_score=('review_score', 'mean')).reset_index()
df = df.merge(rev_agg, on='order_id', how='left')

# Parse dates
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# Convert BRL to DZD
df['subtotal_dzd'] = (df['subtotal'].fillna(0) * BRL_TO_DZD).round(2)
df['payment_dzd'] = (df['payment_value'].fillna(0) * BRL_TO_DZD).round(2)

print(f'Final merged dataset: {df.shape}')
df.head()

## 4. Regional Analysis (Wilaya Simulation)

In [ ]:
# Map Brazilian states to Algerian wilayas
df['wilaya_name'] = df['customer_state'].map(lambda s: STATE_TO_WILAYA.get(s, {'name': 'Other'})['name'])
df['shipping_zone'] = df['customer_state'].map(lambda s: STATE_TO_WILAYA.get(s, {'zone': 'zone_1'})['zone'])

# Regional delivery rates
region_stats = df.groupby(['wilaya_name', 'shipping_zone']).agg(
    total_orders=('order_id', 'count'),
    success_rate=('is_delivered', 'mean'),
    avg_value=('payment_dzd', 'mean'),
).reset_index().sort_values('success_rate')

print('=== Delivery Success Rate by Region ===')
print(region_stats.to_string(index=False))

# Zone-level analysis
zone_stats = df.groupby('shipping_zone').agg(
    orders=('order_id', 'count'),
    success_rate=('is_delivered', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

region_stats.plot(x='wilaya_name', y='success_rate', kind='barh', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('Delivery Success Rate by Wilaya')
axes[0].set_xlabel('Success Rate')
axes[0].axvline(x=df['is_delivered'].mean(), color='red', linestyle='--', label='Average')
axes[0].legend()

zone_stats.plot(x='shipping_zone', y='success_rate', kind='bar', ax=axes[1], color=['green', 'orange', 'red'], legend=False)
axes[1].set_title('Success Rate by Shipping Zone')
axes[1].set_ylabel('Success Rate')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 5. Temporal Analysis

In [ ]:
# Temporal features
df['hour'] = df['order_purchase_timestamp'].dt.hour
df['day_of_week'] = df['order_purchase_timestamp'].dt.dayofweek
df['month'] = df['order_purchase_timestamp'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Hourly pattern
hourly = df.groupby('hour')['is_delivered'].mean()
hourly.plot(kind='line', marker='o', ax=axes[0], color='steelblue')
axes[0].set_title('Delivery Success Rate by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Success Rate')
axes[0].axhline(y=df['is_delivered'].mean(), color='red', linestyle='--', alpha=0.5)

# Day of week pattern
daily = df.groupby('day_of_week')['is_delivered'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily.index = days
daily.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Delivery Success Rate by Day of Week')
axes[1].set_ylabel('Success Rate')
axes[1].tick_params(axis='x', rotation=45)

# Monthly pattern
monthly = df.groupby('month')['is_delivered'].mean()
monthly.plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_title('Delivery Success Rate by Month')
axes[2].set_ylabel('Success Rate')

plt.tight_layout()
plt.show()

print(f'Weekend success rate: {df[df["is_weekend"]==1]["is_delivered"].mean():.3f}')
print(f'Weekday success rate: {df[df["is_weekend"]==0]["is_delivered"].mean():.3f}')

## 6. Product Category Analysis

In [ ]:
# Translate categories
df['category_en'] = df['product_category'].map(
    lambda c: CATEGORY_TRANSLATION.get(str(c), str(c)) if pd.notna(c) else 'unknown'
)

# Top 15 categories by order count
cat_stats = df.groupby('category_en').agg(
    orders=('order_id', 'count'),
    success_rate=('is_delivered', 'mean'),
    avg_value=('payment_dzd', 'mean'),
).reset_index()

top_cats = cat_stats.nlargest(15, 'orders')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_cats.sort_values('success_rate').plot(
    x='category_en', y='success_rate', kind='barh', ax=axes[0], color='steelblue', legend=False
)
axes[0].set_title('Delivery Success Rate by Product Category (Top 15)')
axes[0].set_xlabel('Success Rate')
axes[0].axvline(x=df['is_delivered'].mean(), color='red', linestyle='--')

top_cats.sort_values('orders').plot(
    x='category_en', y='orders', kind='barh', ax=axes[1], color='orange', legend=False
)
axes[1].set_title('Order Count by Product Category (Top 15)')
axes[1].set_xlabel('Order Count')

plt.tight_layout()
plt.show()

## 7. Order Value Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Value distribution by delivery status
df[df['is_delivered']==1]['payment_dzd'].clip(upper=50000).hist(
    bins=50, alpha=0.6, ax=axes[0], label='Delivered', color='green'
)
df[df['is_delivered']==0]['payment_dzd'].clip(upper=50000).hist(
    bins=50, alpha=0.6, ax=axes[0], label='Failed', color='red'
)
axes[0].set_title('Order Value Distribution by Delivery Status (DZD)')
axes[0].set_xlabel('Order Value (DZD)')
axes[0].legend()

# Success rate by value bins
df['value_bin'] = pd.cut(df['payment_dzd'], bins=[0, 1000, 2500, 5000, 10000, 50000, float('inf')],
                         labels=['<1K', '1-2.5K', '2.5-5K', '5-10K', '10-50K', '50K+'])
value_rates = df.groupby('value_bin', observed=True)['is_delivered'].mean()
value_rates.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Success Rate by Order Value Range (DZD)')
axes[1].set_ylabel('Success Rate')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Avg order value (DZD):')
print(f'  Delivered: {df[df["is_delivered"]==1]["payment_dzd"].mean():,.0f} DZD')
print(f'  Failed:    {df[df["is_delivered"]==0]["payment_dzd"].mean():,.0f} DZD')

## 8. Customer History Analysis

In [ ]:
# Repeat customer analysis
customer_orders = df.groupby('customer_unique_id').agg(
    order_count=('order_id', 'count'),
    success_rate=('is_delivered', 'mean'),
    total_spent=('payment_dzd', 'sum'),
).reset_index()

repeat_customers = customer_orders[customer_orders['order_count'] > 1]
single_customers = customer_orders[customer_orders['order_count'] == 1]

print(f'Total unique customers: {len(customer_orders):,}')
print(f'Repeat customers: {len(repeat_customers):,} ({len(repeat_customers)/len(customer_orders):.1%})')
print(f'\nSuccess rate:')
print(f'  First-time buyers: {single_customers["success_rate"].mean():.3f}')
print(f'  Repeat buyers:     {repeat_customers["success_rate"].mean():.3f}')

## 9. Delivery Time Analysis

In [ ]:
# Compute delivery times
df['estimated_days'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days
df['actual_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['delivery_delay'] = df['actual_days'] - df['estimated_days']

delivered = df[df['is_delivered'] == 1].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

delivered['actual_days'].clip(upper=60).hist(bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('Actual Delivery Time (Days)')
axes[0].set_xlabel('Days')
axes[0].axvline(x=delivered['actual_days'].median(), color='red', linestyle='--', label=f'Median: {delivered["actual_days"].median():.0f} days')
axes[0].legend()

delivered['delivery_delay'].clip(-20, 40).hist(bins=40, ax=axes[1], color='orange')
axes[1].set_title('Delivery Delay (Actual - Estimated, Days)')
axes[1].set_xlabel('Days (negative = early, positive = late)')
axes[1].axvline(x=0, color='green', linestyle='--', label='On time')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Median delivery time: {delivered["actual_days"].median():.0f} days')
print(f'Late deliveries: {(delivered["delivery_delay"] > 0).mean():.1%}')

## 10. Feature Correlation Matrix

In [ ]:
# Select numeric features for correlation analysis
corr_features = ['is_delivered', 'payment_dzd', 'n_items', 'hour', 'day_of_week',
                  'month', 'is_weekend', 'estimated_days', 'avg_weight_kg']
corr_df = df[corr_features].dropna()

plt.figure(figsize=(10, 8))
corr = corr_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdYlBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation with Delivery Success')
plt.tight_layout()
plt.show()

print('\nCorrelation with delivery success (is_delivered):')
print(corr['is_delivered'].sort_values(ascending=False).to_string())

## 11. Summary & Key Findings

### Findings to carry into model training:
1. **Delivery success rate**: ~96-97% in Olist (much higher than typical Algerian COD ~60-70%)
2. **Regional variation**: Some states/wilayas have notably different success rates
3. **Temporal patterns**: Hour, day of week, and month all show variation
4. **Product category**: Different categories have different risk profiles
5. **Order value**: Very high value orders may have different risk patterns
6. **Repeat customers**: Show higher success rates than first-time buyers
7. **Delivery time**: Longer estimated delivery times may correlate with risk

### Next step: 
Train the XGBoost risk prediction model in `02_risk_model.ipynb`